# FINAL — Traffic Signs + Helmet + Car Plate OCR
Enable **T4 GPU**, keep input at `MyDrive/DIP/video1.mp4`, then Run all. Full frame is processed (both lanes). Internal person/motorcycle/car boxes are hidden.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/DIP
!git clone -q -b feature/yolo-traffic-safety https://github.com/NVTruong473/DIP.git /content/DIP
%cd /content/DIP/END_DIP
!pip install -q -r requirements.txt


In [ ]:
from pathlib import Path
import os, subprocess, torch
ROOT=Path('/content/drive/MyDrive/DIP'); VIDEO=ROOT/'video1.mp4'; MODELS=ROOT/'models'; OUTPUTS=ROOT/'outputs'
MODELS.mkdir(parents=True,exist_ok=True); OUTPUTS.mkdir(parents=True,exist_ok=True)
os.environ['EASYOCR_MODULE_PATH']=str(MODELS/'easyocr')
assert torch.cuda.is_available(), 'Enable T4 GPU first'
assert VIDEO.exists(), f'Missing {VIDEO}'
print('GPU:',torch.cuda.get_device_name(0)); print('Video:',VIDEO)
subprocess.run(['python','download_models.py','--models-dir',str(MODELS)],check=True)


## Optional one-time helmet fine-tune
Selected dataset: 42,559 real traffic images / ~126k helmet & no-helmet boxes. Leave False for normal inference; set True once to save `helmet_best.pt` permanently in Drive.


In [ ]:
TRAIN_HELMET_ONCE=False
if TRAIN_HELMET_ONCE:
    subprocess.run(['python','training/train_helmet_large.py','--models-dir',str(MODELS),'--dataset-dir',str(ROOT/'datasets/traffic_helmet_42559'),'--runs-dir',str(ROOT/'training_runs'),'--epochs','12','--batch','16'],check=True)
else:
    print('Using cached/pretrained helmet model; no training this run.')


In [ ]:
# RUN ALL 3 TASKS + SAVE TO DRIVE + INLINE PREVIEW
OUT=OUTPUTS/'video1_result.mp4'
subprocess.run(['python','main.py','--input',str(VIDEO),'--output-dir',str(OUTPUTS),'--models-dir',str(MODELS),'--sign-conf','0.32','--scene-conf','0.34','--helmet-conf','0.38','--plate-conf','0.34'],check=True)
assert OUT.exists() and OUT.stat().st_size>0
PREVIEW=Path('/content/video1_result_preview.mp4')
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(OUT),'-vf','scale=960:-2','-c:v','libx264','-crf','29','-pix_fmt','yuv420p','-an',str(PREVIEW)],check=True)
from IPython.display import Video,display
print('Full result:',OUT); print('CSV:',OUTPUTS/'video1_result.csv'); print('Plate crops:',OUTPUTS/'video1_plates')
display(Video(str(PREVIEW),embed=True,width=960,html_attributes='controls'))


# RECOVERY CELL
After Colab disconnect, run **only the cell below**. It restores the repo/runtime and reuses all detector weights already saved in Drive; no retraining.


In [ ]:
# SINGLE RECOVERY / RERUN CELL
import os,sys,subprocess
from pathlib import Path
if not Path('/content/drive/MyDrive').exists():
    from google.colab import drive; drive.mount('/content/drive')
repo=Path('/content/DIP')
if not repo.exists(): subprocess.run(['git','clone','-q','-b','feature/yolo-traffic-safety','https://github.com/NVTruong473/DIP.git',str(repo)],check=True)
work=repo/'END_DIP'; os.chdir(work); subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
root=Path('/content/drive/MyDrive/DIP'); video=root/'video1.mp4'; models=root/'models'; outputs=root/'outputs'; models.mkdir(parents=True,exist_ok=True); outputs.mkdir(parents=True,exist_ok=True)
os.environ['EASYOCR_MODULE_PATH']=str(models/'easyocr')
subprocess.run([sys.executable,'download_models.py','--models-dir',str(models)],check=True)
subprocess.run([sys.executable,'main.py','--input',str(video),'--output-dir',str(outputs),'--models-dir',str(models),'--sign-conf','0.32','--scene-conf','0.34','--helmet-conf','0.38','--plate-conf','0.34'],check=True)
out=outputs/'video1_result.mp4'; preview=Path('/content/video1_result_preview.mp4')
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(out),'-vf','scale=960:-2','-c:v','libx264','-crf','29','-pix_fmt','yuv420p','-an',str(preview)],check=True)
from IPython.display import Video,display
print('Reused models from:',models); print('Saved:',out); display(Video(str(preview),embed=True,width=960,html_attributes='controls'))
